In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

hospital_overview_dataset    = pd.read_csv(PROCESSED_DIR / "hospital_overview_dataset.csv")
patient_flow_dataset         = pd.read_csv(PROCESSED_DIR / "patient_flow_dataset.csv")
department_analytics_dataset = pd.read_csv(PROCESSED_DIR / "department_analytics_dataset.csv")
resource_utilization_dataset = pd.read_csv(PROCESSED_DIR / "resource_utilization_dataset.csv")

In [2]:
for df, date_cols in [
    (hospital_overview_dataset, ['admission_date', 'discharge_date']),
    (patient_flow_dataset, ['event_date']),
    (department_analytics_dataset, ['date']),
    (resource_utilization_dataset, ['date']),
]:
    for c in date_cols:
        df[c] = pd.to_datetime(df[c])

issues = []  # (table, check, detail) for every FAIL, used for the final summary


def flag(table, check, ok, detail):
    print(f"[{'PASS' if ok else 'FAIL'}] {table} :: {check} -- {detail}")
    if not ok:
        issues.append((table, check, detail))


# =====================================================================
# 1. PRIMARY KEY UNIQUENESS
# =====================================================================
print("=== 1. PRIMARY KEY UNIQUENESS ===")

flag("hospital_overview", "admission_id unique & non-null",
     hospital_overview_dataset['admission_id'].is_unique and hospital_overview_dataset['admission_id'].notna().all(),
     f"{hospital_overview_dataset['admission_id'].nunique()} unique / {len(hospital_overview_dataset)} rows")

flag("patient_flow", "flow_event_id unique & non-null",
     patient_flow_dataset['flow_event_id'].is_unique and patient_flow_dataset['flow_event_id'].notna().all(),
     f"{patient_flow_dataset['flow_event_id'].nunique()} unique / {len(patient_flow_dataset)} rows")

dep_dupes = department_analytics_dataset.duplicated(subset=['department_id', 'date']).sum()
flag("department_analytics", "(department_id, date) composite key unique",
     dep_dupes == 0, f"{dep_dupes} duplicate (department_id, date) pairs")

flag("resource_utilization", "resource_utilization_id unique & non-null",
     resource_utilization_dataset['resource_utilization_id'].is_unique and
     resource_utilization_dataset['resource_utilization_id'].notna().all(),
     f"{resource_utilization_dataset['resource_utilization_id'].nunique()} unique / {len(resource_utilization_dataset)} rows")

for rtype, keys in [('Bed', ['department_id', 'date']), ('Staff', ['ward_id', 'shift']), ('Drug Inventory', ['drug_id'])]:
    block = resource_utilization_dataset[resource_utilization_dataset['resource_type'] == rtype]
    dupes = block.duplicated(subset=keys).sum()
    flag("resource_utilization", f"{rtype} natural key {keys} unique",
         dupes == 0, f"{dupes} duplicates among {len(block)} {rtype} rows")


# =====================================================================
# 2. GRAIN SANITY
# =====================================================================
print("\n=== 2. GRAIN SANITY ===")

flag("hospital_overview", "grain = 1 row per admission",
     len(hospital_overview_dataset) == hospital_overview_dataset['admission_id'].nunique(),
     f"{len(hospital_overview_dataset)} rows, {hospital_overview_dataset['admission_id'].nunique()} distinct admissions")

n_dx_events = (patient_flow_dataset['event_type'] == 'Diagnostic Test').sum()
expected_flow_rows = hospital_overview_dataset['admission_id'].nunique() * 2 + n_dx_events
flag("patient_flow", "row count = 2 events/admission + 1 event/diagnostic test",
     len(patient_flow_dataset) == expected_flow_rows,
     f"{len(patient_flow_dataset)} actual vs {expected_flow_rows} expected")

paired = patient_flow_dataset[patient_flow_dataset['event_type'].isin(['Admission', 'Discharge'])]
unpaired = (paired.groupby('admission_id')['event_type'].nunique() != 2).sum()
flag("patient_flow", "every admission has exactly 1 Admission + 1 Discharge event",
     unpaired == 0, f"{unpaired} admissions missing a paired event")

gap_depts = 0
for _, g in department_analytics_dataset.groupby('department_id'):
    full_range = pd.date_range(g['date'].min(), g['date'].max(), freq='D')
    if len(g) != len(full_range):
        gap_depts += 1
flag("department_analytics", "daily calendar spine has no gaps, per department",
     gap_depts == 0, f"{gap_depts} department(s) with missing calendar days")

bed_rows = (resource_utilization_dataset['resource_type'] == 'Bed').sum()
expected_bed_rows = (department_analytics_dataset['total_beds'] > 0).sum()
flag("resource_utilization", "Bed block row count matches department_analytics beds-present rows",
     bed_rows == expected_bed_rows, f"{bed_rows} vs {expected_bed_rows}")


# =====================================================================
# 3. VALUE-RANGE VALIDITY  (per project doc's Outlier Validation step)
# =====================================================================
print("\n=== 3. VALUE-RANGE VALIDITY ===")


def range_check(table, df, col, lo=None, hi=None):
    s = df[col].dropna()
    bad = pd.Series(False, index=s.index)
    if lo is not None:
        bad |= s < lo
    if hi is not None:
        bad |= s > hi
    flag(table, f"{col} within [{lo}, {hi}]", bad.sum() == 0, f"{bad.sum()} out-of-range value(s)")


range_check("hospital_overview", hospital_overview_dataset, 'patient_age_at_admission', 0, 120)
range_check("hospital_overview", hospital_overview_dataset, 'length_of_stay_days', 0, None)
range_check("hospital_overview", hospital_overview_dataset, 'total_bill_amount', 0, None)
range_check("hospital_overview", hospital_overview_dataset, 'insurance_covered_amount', 0, None)
range_check("hospital_overview", hospital_overview_dataset, 'patient_payable_amount', 0, None)

range_check("patient_flow", patient_flow_dataset, 'patient_age_at_admission', 0, 120)
range_check("patient_flow", patient_flow_dataset, 'length_of_stay_days', 0, None)

range_check("department_analytics", department_analytics_dataset, 'bed_occupancy_rate_pct', 0, 100)
range_check("department_analytics", department_analytics_dataset, 'readmission_rate_pct', 0, 100)
range_check("department_analytics", department_analytics_dataset, 'avg_length_of_stay_days', 0, None)
range_check("department_analytics", department_analytics_dataset, 'department_efficiency_score', 0, 100)
range_check("department_analytics", department_analytics_dataset, 'admissions_count', 0, None)
range_check("department_analytics", department_analytics_dataset, 'discharges_count', 0, None)

range_check("resource_utilization", resource_utilization_dataset, 'utilization_rate_pct', 0, 100)
range_check("resource_utilization", resource_utilization_dataset, 'units_in_use', 0, None)


# =====================================================================
# 4. CROSS-TABLE KEY CONSISTENCY
# =====================================================================
print("\n=== 4. CROSS-TABLE KEY CONSISTENCY ===")

ho_ids = set(hospital_overview_dataset['admission_id'])
pf_ids = set(patient_flow_dataset['admission_id'].dropna())
flag("hospital_overview <-> patient_flow", "admission_id sets match exactly",
     ho_ids == pf_ids, f"{len(ho_ids - pf_ids)} in overview only, {len(pf_ids - ho_ids)} in flow only")

ho_depts = set(hospital_overview_dataset['department_id'])
da_depts = set(department_analytics_dataset['department_id'])
flag("hospital_overview <-> department_analytics", "every overview department_id exists in analytics",
     ho_depts.issubset(da_depts), f"{len(ho_depts - da_depts)} missing department_id(s)")

ru_depts = set(resource_utilization_dataset['department_id'].dropna())
flag("resource_utilization <-> department_analytics", "every resource department_id exists in analytics",
     ru_depts.issubset(da_depts), f"{len(ru_depts - da_depts)} missing department_id(s)")

total_adm = department_analytics_dataset['admissions_count'].sum()
flag("hospital_overview <-> department_analytics", "sum(admissions_count) matches admission row count",
     total_adm == len(hospital_overview_dataset), f"{total_adm} vs {len(hospital_overview_dataset)}")

total_disch = department_analytics_dataset['discharges_count'].sum()
flag("hospital_overview <-> department_analytics", "sum(discharges_count) matches admission row count",
     total_disch == len(hospital_overview_dataset), f"{total_disch} vs {len(hospital_overview_dataset)}")

total_readm_da = department_analytics_dataset['readmission_count'].sum()
total_readm_ho = hospital_overview_dataset['readmission_flag'].sum()
flag("hospital_overview <-> department_analytics", "sum(readmission_count) matches sum(readmission_flag)",
     total_readm_da == total_readm_ho, f"{total_readm_da} vs {total_readm_ho}")


# =====================================================================
# 5. COMPLETENESS SCORE  (project doc target: >95% completeness / <2% missing)
# Computed directly from the data -- not assumed.
# =====================================================================
print("\n=== 5. COMPLETENESS SCORE ===")

# Columns that are EXPECTED to be partially null BY DESIGN (documented in the
# table-build step) -- excluded from the score, reported separately instead.
documented_nullable = {
    'hospital_overview': ['primary_doctor_id', 'readmission_gap_days'],
    'patient_flow': ['ward_id', 'ward_name', 'ward_type', 'bed_id', 'length_of_stay_days',
                      'doctor_id', 'doctor_name', 'doctor_specialization',
                      'test_name', 'test_category', 'result_status'],
    'department_analytics': ['bed_occupancy_rate_pct', 'readmission_rate_pct', 'avg_length_of_stay_days',
                              'staff_to_patient_ratio', 'department_efficiency_score'],
    'resource_utilization': ['date', 'ward_id', 'ward_name', 'shift', 'total_units_available', 'utilization_rate_pct',
                              'doctor_count', 'nurse_count', 'technician_count', 'pharmacist_count', 'admin_count',
                              'drug_id', 'drug_name', 'drug_category', 'reorder_level_threshold', 'shortage_flag',
                              'manufacturer_name', 'reliability_rating'],
}


def completeness_report(name, df):
    exception_cols = [c for c in documented_nullable.get(name, []) if c in df.columns]
    core_cols = [c for c in df.columns if c not in exception_cols]
    core_cells = df[core_cols].size
    core_missing = int(df[core_cols].isna().sum().sum())
    core_completeness = 100 * (1 - core_missing / core_cells)

    print(f"\n{name}:")
    print(f"  Core-field completeness = 1 - ({core_missing} missing / {core_cells} cells, "
          f"{len(core_cols)} required columns) = {core_completeness:.2f}%")
    print(f"  -> {'PASS' if core_completeness >= 95 else 'FAIL'} "
          f"(target: >95% complete / <2% missing -> missing = {100 - core_completeness:.2f}%)")
    for c in exception_cols:
        print(f"  [documented exception, excluded from score] {c}: {100 * df[c].isna().mean():.1f}% null")
    return core_completeness


scores = {name: completeness_report(name, df) for name, df in [
    ('hospital_overview', hospital_overview_dataset),
    ('patient_flow', patient_flow_dataset),
    ('department_analytics', department_analytics_dataset),
    ('resource_utilization', resource_utilization_dataset),
]}


# =====================================================================
# FINAL SUMMARY
# =====================================================================
print("\n" + "=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)
if issues:
    print(f"{len(issues)} check(s) FAILED:")
    for t, c, d in issues:
        print(f"  - [{t}] {c}: {d}")
else:
    print("All structural checks PASSED.")

print("\nCompleteness scores (core, required fields only):")
for name, score in scores.items():
    print(f"  {name:25s} {score:6.2f}%  {'OK' if score >= 95 else 'BELOW 95% TARGET'}")

=== 1. PRIMARY KEY UNIQUENESS ===
[PASS] hospital_overview :: admission_id unique & non-null -- 45000 unique / 45000 rows
[PASS] patient_flow :: flow_event_id unique & non-null -- 153269 unique / 153269 rows
[PASS] department_analytics :: (department_id, date) composite key unique -- 0 duplicate (department_id, date) pairs
[PASS] resource_utilization :: resource_utilization_id unique & non-null -- 13549 unique / 13549 rows
[PASS] resource_utilization :: Bed natural key ['department_id', 'date'] unique -- 0 duplicates among 13224 Bed rows
[PASS] resource_utilization :: Staff natural key ['ward_id', 'shift'] unique -- 0 duplicates among 75 Staff rows
[PASS] resource_utilization :: Drug Inventory natural key ['drug_id'] unique -- 0 duplicates among 250 Drug Inventory rows

=== 2. GRAIN SANITY ===
[PASS] hospital_overview :: grain = 1 row per admission -- 45000 rows, 45000 distinct admissions
[PASS] patient_flow :: row count = 2 events/admission + 1 event/diagnostic test -- 153269 actual v